In [10]:
import pandas as pd

msk = pd.read_csv(r"datasets\msk_chord_2024\msk_chord_out\nsclc_patient_level_features.csv")
tcga = pd.read_csv("datasets/tcga_nsclc_clinical_clean.csv")

print(msk.shape, tcga.shape)

(7809, 53) (1026, 27)


In [11]:
# TCGA tobacco_smoking_history codes (standard TCGA coding):
# 1 = Lifelong Non-smoker, 2 = Current smoker, 3/4/5 = Current reformed smoker (various durations)
tcga_smoking_map = {
    1: "Never",
    2: "Former/Current Smoker",
    3: "Former/Current Smoker",
    4: "Former/Current Smoker",
    5: "Former/Current Smoker",
}
tcga["SMOKING_STATUS"] = tcga["tobacco_smoking_history"].map(tcga_smoking_map).fillna("Unknown")
msk["SMOKING_STATUS"] = msk["SMOKING_PREDICTIONS_3_CLASSES"].fillna("Unknown")

print(tcga["SMOKING_STATUS"].value_counts())
print(msk["SMOKING_STATUS"].value_counts())

SMOKING_STATUS
Former/Current Smoker    907
Never                     93
Unknown                   26
Name: count, dtype: int64
SMOKING_STATUS
Former/Current Smoker    5438
Never                    2027
Unknown                   344
Name: count, dtype: int64


In [12]:
def collapse_stage(s):
    if pd.isna(s):
        return "Unknown"
    s = str(s)
    if "IV" in s:
        return "Stage 4"
    elif any(x in s for x in ["I", "II", "III"]):
        return "Stage 1-3"
    return "Unknown"

tcga["STAGE_COLLAPSED"] = tcga["pathologic_stage"].apply(collapse_stage)
msk["STAGE_COLLAPSED"] = msk["STAGE_HIGHEST_RECORDED"].fillna("Unknown")

print(tcga["STAGE_COLLAPSED"].value_counts())
print(msk["STAGE_COLLAPSED"].value_counts())

STAGE_COLLAPSED
Stage 1-3    981
Stage 4       33
Unknown       12
Name: count, dtype: int64
STAGE_COLLAPSED
Stage 1-3    4387
Stage 4      3421
Unknown         1
Name: count, dtype: int64


In [13]:
# TCGA days -> months, to match MSK-CHORD's OS_MONTHS
tcga["OS_MONTHS"] = tcga["days_to_event"] / 30.44

# Align column names for the fields both datasets share conceptually
msk_harmonized = msk.rename(columns={
    "CURRENT_AGE_DEID": "AGE",
    "GENDER": "SEX",
    "RACE": "RACE",
    "ETHNICITY": "ETHNICITY",
    "CANCER_TYPE_DETAILED": "HISTOLOGY_DETAILED",
})[[
    "PATIENT_ID", "AGE", "SEX", "RACE", "ETHNICITY", "SMOKING_STATUS",
    "STAGE_COLLAPSED", "HISTOLOGY_DETAILED", "OS_MONTHS", "OS_STATUS",
    "ever_pdl1_positive", "num_treatment_events", "num_distinct_tumor_sites",
]].copy()
msk_harmonized["SOURCE_DATASET"] = "MSK-CHORD"

tcga_harmonized = tcga.rename(columns={
    "bcr_patient_barcode": "PATIENT_ID",
    "age_at_initial_pathologic_diagnosis": "AGE",
    "gender": "SEX",
    "race": "RACE",
    "ethnicity": "ETHNICITY",
    "histological_type": "HISTOLOGY_DETAILED",
})[[
    "PATIENT_ID", "AGE", "SEX", "RACE", "ETHNICITY", "SMOKING_STATUS",
    "STAGE_COLLAPSED", "HISTOLOGY_DETAILED", "OS_MONTHS", "OS_STATUS",
    "eastern_cancer_oncology_group", "karnofsky_performance_score",
]].copy()
tcga_harmonized["SOURCE_DATASET"] = "TCGA"

print(msk_harmonized.shape, tcga_harmonized.shape)

(7809, 14) (1026, 13)


In [14]:
combined = pd.concat([msk_harmonized, tcga_harmonized], ignore_index=True)
print(combined.shape)
print(combined["SOURCE_DATASET"].value_counts())
print(combined["SEX"].value_counts(dropna=False))  # check for encoding mismatches (e.g. "Male" vs "MALE")

combined.to_csv("datasets/combined_clinical.csv", index=False)

(8835, 16)
SOURCE_DATASET
MSK-CHORD    7809
TCGA         1026
Name: count, dtype: int64
SEX
Female     4567
Male       3241
MALE        615
FEMALE      411
Unknown       1
Name: count, dtype: int64


In [15]:
combined["SEX"] = combined["SEX"].str.strip().str.title()
print(combined["SEX"].value_counts(dropna=False))

SEX
Female     4978
Male       3856
Unknown       1
Name: count, dtype: int64


In [16]:
# If NaN genuinely means "no treatment events recorded" for MSK-CHORD patients specifically:
msk_mask = combined["SOURCE_DATASET"] == "MSK-CHORD"
combined.loc[msk_mask, "num_treatment_events"] = combined.loc[msk_mask, "num_treatment_events"].fillna(0)

In [17]:
combined.to_csv("datasets/combined_clinical.csv", index=False)
print(combined.shape)
print(combined.isna().sum())

(8835, 16)
PATIENT_ID                          0
AGE                                29
SEX                                 0
RACE                              180
ETHNICITY                         303
SMOKING_STATUS                      0
STAGE_COLLAPSED                     0
HISTOLOGY_DETAILED                  1
OS_MONTHS                          15
OS_STATUS                           0
ever_pdl1_positive               4421
num_treatment_events             1026
num_distinct_tumor_sites         1410
SOURCE_DATASET                      0
eastern_cancer_oncology_group    8341
karnofsky_performance_score      8532
dtype: int64
